<a href="https://colab.research.google.com/github/caasalazarsa/SistemasOperativos/blob/main/MapTest.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install flask_ngrok
!pip install pyngrok
!pip install paho-mqtt folium flask

In [ ]:
import getpass
import os
import threading
from pyngrok import ngrok, conf
import folium
from flask import Flask, jsonify, render_template_string
import paho.mqtt.client as mqtt
import requests

conf.get_default().auth_token = getpass.getpass("Enter ngrok auth token: ")
# Open a ngrok tunnel to the HTTP server
public_url = ngrok.connect(5000).public_url
print(f" * ngrok tunnel \"{public_url}\" -> \"http://127.0.0.1:5000/\"")
# Variables globales para la latitud y longitud
latitude = 0.0
longitude = 0.0

def on_connect(client, userdata, flags, rc):
    client.subscribe("latlong")

def on_message(client, userdata, msg):
    global latitude, longitude
    try:
        lat, lon = map(float, msg.payload.decode().split(","))
        latitude = lat
        longitude = lon
    except ValueError:
        print("Received invalid latitude/longitude data")

client = mqtt.Client()
client.on_connect = on_connect
client.on_message = on_message

try:
    client.connect("broker.mqtt.cool", 1883, 60)
    client.loop_start()
except Exception as e:
    print(f"Could not connect to MQTT broker: {e}")



app = Flask(__name__)



@app.route('/')
def index():
    map_ = folium.Map(location=[latitude, longitude], zoom_start=15)
    folium.Marker([latitude, longitude]).add_to(map_)

    # HTML para el mapa
    html = map_.get_root().render() + '''
    <script>
        let marker;

        function initMap() {
            marker = L.marker([0, 0]).addTo(map);
        }

        function updateMap(lat, lon) {
            marker.setLatLng([lat, lon]);
            location.reload();
        }


        setInterval(updateMap, 1000);
    </script>
    <div id="map" style="height: 500px;"></div>
    '''
    return html



threading.Thread(target=app.run, kwargs={"use_reloader": False}).start()

INFO:werkzeug:127.0.0.1 - - [31/Oct/2024 15:18:54] "GET /coordinates HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [31/Oct/2024 15:18:54] "GET /coordinates HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [31/Oct/2024 15:18:55] "GET /coordinates HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [31/Oct/2024 15:18:56] "GET /coordinates HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [31/Oct/2024 15:18:57] "GET /coordinates HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [31/Oct/2024 15:18:58] "GET /coordinates HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [31/Oct/2024 15:18:59] "GET /coordinates HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [31/Oct/2024 15:19:00] "GET /coordinates HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [31/Oct/2024 15:19:01] "GET /coordinates HTTP/1.1" 200 -


Enter ngrok auth token: ··········
 * ngrok tunnel "https://f4a6-34-139-28-79.ngrok-free.app" -> "http://127.0.0.1:5000/"


<ipython-input-7-8f9ab9623e21>:30: DeprecationWarning: Callback API version 1 is deprecated, update to latest version
  client = mqtt.Client()


 * Serving Flask app '__main__'
